In [1]:
import torch
from türkçe_dil_modelim.tokenizer import Tokenizer
from türkçe_dil_modelim.model import Model

tokenizer= Tokenizer("türkçe_dil_modelim/tokenizer.json")
prompt="Benim adım Burak"
prompts= [
    "Benim adım Burak",
    "senin adın ",
    "nerede yaşıyorsun"
]# eğer batch batch eğitmek istiyorsam bu şekilde vermem gerekiyo verileri.
#mesela resim verisinde bu durum her bir resim özelinde 2D bir liste. cümle için ise 1d bir liste oluyor
tokens = tokenizer.encode(prompts[0])
tokens.to("cpu")


tensor([1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
        1153])

In [2]:
batch_tokens= tokenizer.encode_batch(prompts,32)

In [3]:
batch_tokens.shape

torch.Size([3, 32])

In [4]:
batch_tokens

tensor([[1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
         1153,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 497,  622,  373,  342, 1148,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 534, 1162,  462,  622,  371,  297,  425,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627]])

In [5]:
context_length= 32
torch.manual_seed(1)

model= Model(vocab_size=tokenizer.get_vocab_size(),embedding_dim=12,num_heads=4,context_length=context_length,num_layers=8,device="cpu")

In [6]:
out= model(batch_tokens)
out=out.squeeze(0)
out.shape

torch.Size([3, 32, 1376])

In [10]:
out=model.generate(tokens,2, temperature=10)
tokenizer.decode(out)

'Benim adım Burakıyazeka'

In [13]:
with open("türkçe_dil_modelim/text.txt","r") as f:
    text= f.read()

len(text),text[:100]

(1094,
 'Yapay zekâ sistemleri, insan düşünme biçimini taklit etmeye çalışırken aslında çok daha temel bir pr')

In [14]:
token_ids= tokenizer.encode(text)
len(token_ids), type(token_ids)

(693, torch.Tensor)

In [22]:
from türkçe_dil_modelim.text_dataset import create_data_loader

stride=12

train_data_loader= create_data_loader(token_ids=token_ids.tolist(),context_length=context_length,stride=stride,batch_size=3,shuffle=True)

In [17]:
parameters_count= sum(p.numel() for p in model.parameters())
print(parameters_count)

print(model)

55762
Model(
  (embedding): Embedding(
    (embedding): Embedding(1378, 12)
  )
  (layers): Sequential(
    (0): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (layer_norm1): LayerNorm()
      (layer_norm2): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=48, bias=True)
        (up_proj): Linear(in_features=12, out_features=48, bias=True)
        (down_proj): Linear(in_features=48, out_features=12, bias=True)
        (gelu): GELU()
      )
    )
    (1): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_feat

In [20]:
out= model(batch_tokens)
out,out.flatten(),out.flatten(0,1),out.flatten(0,1).shape

(tensor([[[ 0.4469, -0.0071, -0.8129,  ...,  0.2220, -0.3666,  0.7874],
          [-0.2389, -0.7487,  0.9235,  ...,  1.1386, -0.5788,  1.0173],
          [-0.0061,  0.3511, -0.0983,  ...,  0.5000,  0.2660,  1.1216],
          ...,
          [ 1.0076,  0.0336, -0.9433,  ...,  0.4603,  0.8066,  1.1032],
          [ 0.8151,  0.0252, -0.7139,  ...,  0.4016,  1.0135,  0.7093],
          [ 0.5656,  0.0431, -0.3709,  ...,  0.4314,  1.0436,  0.3284]],
 
         [[ 0.7468,  0.3204, -0.7823,  ..., -0.2027,  0.2968,  1.3997],
          [ 1.1291, -0.7942, -0.4345,  ...,  0.7802, -0.3155,  1.2360],
          [ 0.4317,  0.0990, -0.9312,  ...,  0.4478,  0.0998,  0.9298],
          ...,
          [ 1.0076,  0.0336, -0.9433,  ...,  0.4603,  0.8066,  1.1032],
          [ 0.8151,  0.0252, -0.7139,  ...,  0.4016,  1.0135,  0.7093],
          [ 0.5656,  0.0431, -0.3709,  ...,  0.4314,  1.0436,  0.3284]],
 
         [[-0.0474,  0.1036, -0.0059,  ...,  1.1120, -0.1202,  1.6819],
          [ 0.1667,  0.0245,

In [24]:
for i , (X,Y) in enumerate(train_data_loader):
    print(X.shape,Y.shape,Y.flatten().shape)
    break

torch.Size([3, 32]) torch.Size([3, 32]) torch.Size([96])


In [26]:
import torch.nn as nn 
device="cpu"
loss_fn= nn.CrossEntropyLoss()

optimizer= torch.optim.AdamW(model.parameters(),lr=1e-3)

epochs= 20

for epoch in range(epochs):
    total_loss=0
    for i, (X,Y) in enumerate(train_data_loader):
        X= X.to(device)
        Y= Y.to(device)

        pred= model(X)
        loss= loss_fn(pred.flatten(0,1),Y.flatten())
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    average_loss= total_loss/ len(train_data_loader)
    print(f"Epoch {epoch+1} loss: {loss.item()} average_loss: {average_loss}")

Epoch 1 loss: 6.95994234085083 average_loss: 7.241826559367933
Epoch 2 loss: 6.081425666809082 average_loss: 6.654305558455618
Epoch 3 loss: 5.653647422790527 average_loss: 5.800693235899272
Epoch 4 loss: 4.830067157745361 average_loss: 5.102265408164577
Epoch 5 loss: 4.342641830444336 average_loss: 4.693447564777575
Epoch 6 loss: 4.6052350997924805 average_loss: 4.432196165385999
Epoch 7 loss: 4.459949016571045 average_loss: 4.252635579360159
Epoch 8 loss: 4.046427249908447 average_loss: 4.1329356620186255
Epoch 9 loss: 3.8141162395477295 average_loss: 4.051800627457468
Epoch 10 loss: 4.107280254364014 average_loss: 4.003473746149163
Epoch 11 loss: 3.7800934314727783 average_loss: 3.9524101332614294
Epoch 12 loss: 3.7923660278320312 average_loss: 3.911695869345414
Epoch 13 loss: 4.020132064819336 average_loss: 3.877518553482859
Epoch 14 loss: 3.5505189895629883 average_loss: 3.836064853166279
Epoch 15 loss: 3.60558819770813 average_loss: 3.8028896984301115
Epoch 16 loss: 3.65723490715

In [ ]:
#temperature , top_k , top_p


In [10]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor, dim=-1)

_, max_index = torch.max(probs, dim=-1)
next_token = max_index.item()

In [11]:
probs.squeeze(0) # her zaman en yüksek olasılıklıyı seçmesin diye temperature, top_k, top_p gibi parametreleri de kullanıcaz

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [12]:
next_token

0

In [13]:
probs= probs.squeeze(0)

In [14]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [15]:
import torch

# 3 token için olasılık dağılımı
probs = torch.tensor([0.1, 0.3, 0.6])

# 1 token seç
sample = torch.multinomial(probs, 1)


counts= [0,0,0]
for i in range(100000): # bu kısımda deneme sayısı arttıkça gerçek olasılıklarıyla aynı değer olmaya yakınlaşacak çıktı
    sample= torch.multinomial(probs,1).item()
    counts[sample]+=1
print(counts)

[9964, 30021, 60015]


[[[1179,
   372,
   1148,
   353,
   622,
   373,
   342,
   1152,
   622,
   1179,
   323,
   373,
   1153,
   179,
   305]]]

In [19]:
out_tensor

tensor([[1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
          323.,  373., 1153.,  179.,  305.]])

In [ ]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor/1.5, dim=-1) # temperature denemesi

probs= probs.squeeze(0)

sample_count={}
for _ in range(10000):
    sample= torch.multinomial(probs,1)
    sample_count[sample.item()]= sample_count.get(sample.item(),0)+1
sample_count

{9: 4948, 0: 5052}

In [24]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [35]:
temperature=0.1 # temperature değeri 1'den büyük olduğunda olasılıklar birbirine yaklaştığından modelin farklı eyler deme olasılığı artar
#1'den küçükse model daha determenistik olur
out_tensor = torch.tensor([out], dtype=torch.float32)

adjusted_out= out_tensor/temperature
torch.softmax(adjusted_out,dim=-1)

tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [32]:
outs= {}

for _ in range (100):
    response= model.generate(tokens,max_new_tokens=2,temperature=0.4)
    decode= tokenizer.decode(response)
    outs[decode]= outs.get(decode,0) +1

outs


{'Benim adım Burakbotanikakısa': 1,
 'Benim adım Burakaşçıfibroyad': 1,
 'Benim adım Burakhikingçarşamba': 1,
 'Benim adım Burakkırıküyorlar': 1,
 'Benim adım BurakfiltirlemekTennis': 1,
 'Benim adım Buraküyorumcuma': 1,
 'Benim adım Burakdamatavukat': 1,
 'Benim adım BurakAmazonparlak': 1,
 'Benim adım Buraktemizlemekooforit': 1,
 'Benim adım Burakcı': 1,
 'Benim adım Burakflüoresanlaşmaküyle': 1,
 'Benim adım Burakbahçeiğne': 1,
 'Benim adım Burakkapıçözelmek': 1,
 'Benim adım Buraküyorumsarcoma': 1,
 'Benim adım Burakmicroservicesgüzel': 1,
 'Benim adım Burak.var': 1,
 'Benim adım Buraketgöstermek': 1,
 'Benim adım BurakamcaHybrid': 1,
 'Benim adım Burakuyorsunuzmi': 1,
 'Benim adım Burakşarkıaids': 1,
 'Benim adım Burakneredenkoyu': 1,
 'Benim adım BurakyaniParkinson': 1,
 'Benim adım BurakkalkamkAmerica': 1,
 'Benim adım Burakyanmaokul': 1,
 'Benim adım Buraksmsh': 1,
 'Benim adım Burakheykeltıraşlıkkilit': 1,
 'Benim adım Burakhikinghayvancılık': 1,
 'Benim adım Burakaşçısebze': 

In [39]:
out_tensor

tensor([1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
         323.,  373., 1153.,  311.,   59.])

In [43]:
sorted_outs= sorted(out_tensor.tolist(), reverse=True)
sorted_indexes=[]
top_k=10
temperature= 0.5
for so in sorted_outs[:top_k]:
    so_index= out_tensor.tolist().index(so)
    sorted_indexes.append(so_index)
sorted_outs= torch.tensor(sorted_outs[:top_k])
adjusted_out= torch.softmax(sorted_outs/temperature,dim=-1)
sorted_outs, sorted_indexes, adjusted_out

(tensor([1179., 1179., 1153., 1152., 1148.,  622.,  622.,  373.,  373.,  372.]),
 [0, 0, 12, 7, 2, 4, 4, 5, 5, 1],
 tensor([5.0000e-01, 5.0000e-01, 1.3051e-23, 1.7663e-24, 5.9253e-28, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]))